# NB1 — Demand Forecasting & Delivery-Risk Scores

**Scope.** Produces two consumer parquets under `outputs/`:
- `nb1_weekly_order_volume.parquet` — weekly order count per seller (consumed by NB2's lead-indicator section).
- `nb1_seller_demand_scores.parquet` — per-seller forecast uplift, average delivery delay, and delay-risk flag (feeds the convergence layer in NB3 / the main notebook).

**Rubric surface in this notebook.** RDDs · DataFrames · SparkSQL (≥3 queries on temp views) · ML Pipelines · MLlib (`GBTRegressor` + `RandomForestRegressor` under `CrossValidator(folds=3)` with `RegressionEvaluator(rmse)`). Streaming is bonus-only (skipped — the three required parquets are in place).

**Big-data hygiene enforced throughout.** Explicit `StructType` schemas via `loaders.load_*`; `broadcast()` on the small sellers / products lookups; `@step` cache so reruns on unchanged inputs + code skip the compute; `approxQuantile` / `approx_count_distinct` for EDA; every `orderBy` paired with a `limit`.

**Thin-wrapper notice.** This notebook is a report surface — every Spark primitive is called via `src/olist/pipeline/demand.py` and inspected inline with `inspect.getsource(...)`, `.getStages()`, `.explainParams()`, or `.show()`. No transformation logic lives in notebook cells.


## 1. Boot — `SparkSession` + `JAVA_HOME`

The local Homebrew OpenJDK 11 path is exported before PySpark imports so the Py4J bridge can launch the JVM. `shuffle_partitions=64`, driver memory 6 GB, time zone `America/Sao_Paulo` — configured in `src/olist/spark_session.py`.

In [1]:
import os, sys, inspect
from pathlib import Path

os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@11/libexec/openjdk.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from olist.spark_session import get_spark

spark = get_spark("nb1-demand-forecasting")
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version, "| driver python:", sys.executable)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 13:34:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.8 | driver python: /Users/lukas/Desktop/NOVA-IMS_Second_Semester/Big Data Analysis/.venv/bin/python


## 2. RDD warm-up — `textFile → filter/map/reduceByKey → typed DataFrame`

Reads `data/olist_orders_dataset.csv` as a raw text RDD, strips the header, maps each row to `(purchase_date, 1)`, filters bad rows, reduces by key, and rebuilds a typed DataFrame with an explicit schema. The chain itself is visible below via `inspect.getsource(...)` — no duplicated logic, single source of truth in `src/olist/pipeline/demand.py`.

In [2]:
from olist.pipeline.demand import rdd_daily_order_count

print(inspect.getsource(rdd_daily_order_count))


def rdd_daily_order_count(spark: SparkSession) -> DataFrame:
    """Demonstrate the required RDD primitive: read the raw orders CSV as text,
    split / filter / reduceByKey into a (date → order-count) pair, and rebuild
    a typed DataFrame with an explicit schema.

    The returned DataFrame is only used for a sanity check against the typed
    loader — downstream work reads the typed DataFrame via `loaders.load_orders`.
    """
    data_dir = Path(__file__).resolve().parents[3] / "data"
    orders_csv = str(data_dir / "olist_orders_dataset.csv")
    raw = spark.sparkContext.textFile(orders_csv)
    header = raw.first()

    def _parse(line: str):
        try:
            parts = line.split(",")
            ts = parts[3]  # order_purchase_timestamp
            if not ts:
                return None
            return (datetime.strptime(ts[:10], "%Y-%m-%d").date(), 1)
        except Exception:
            return None

    daily_pairs = (
        raw.filter(lambda row: row != header)
 

In [3]:
daily_orders_rdd_df = rdd_daily_order_count(spark)
print("RDD-derived daily rows:", daily_orders_rdd_df.count())
daily_orders_rdd_df.orderBy("purchase_date").limit(5).show()


RDD-derived daily rows: 634


+-------------+-----------+
|purchase_date|order_count|
+-------------+-----------+
|   2016-09-04|          1|
|   2016-09-05|          1|
|   2016-09-13|          1|
|   2016-09-15|          1|
|   2016-10-02|          1|
+-------------+-----------+



## 3. Typed loads — explicit schemas via `loaders.load_*`

Every CSV is loaded with a pre-declared `StructType` from `src/olist/schemas.py` — no `inferSchema=True`, so there is no extra full-file pass. Row counts are an audit trail logged in `docs/decisions_log.md`.

In [4]:
from olist.pipeline.demand import load_core_tables

tables = load_core_tables(spark)
for name, df in tables.items():
    print(f"{name:>24}: {df.count():>8,}")


                  orders:   99,441
             order_items:  112,650


               customers:   99,441
                 sellers:    3,095
                products:   32,951


           order_reviews:  104,162
    category_translation:       71


## 4. Geolocation centroids — aggregate 1 M-row geolocation to one row per zip

The raw `olist_geolocation_dataset.csv` has many rows per zip (one per address). We reduce to the centroid once and broadcast the result (≤10 MB) to every join that needs state-level location. `@step` caches the parquet so reruns skip this entirely — cold-run takes ~10 s; warm-run is a parquet read.

In [5]:
from olist.pipeline.demand import build_geo_centroids

geo_centroids = build_geo_centroids(spark)
print("geo_centroids rows:", geo_centroids.count())
geo_centroids.limit(3).show()


[cache] run demand.geo_centroids (3.5s, wrote 19015 rows → outputs/geo_centroids.parquet)
geo_centroids rows: 19015
+---------------------------+-------------------+-------------------+-----+
|geolocation_zip_code_prefix|                lat|                lng|state|
+---------------------------+-------------------+-------------------+-----+
|                       1001|-23.550189776551765|-46.634023555904214|   SP|
|                       1002| -23.54814573176355| -46.63497921074498|   SP|
|                       1003|-23.548993724813155| -46.63573130997587|   SP|
+---------------------------+-------------------+-------------------+-----+



## 5. Filter to delivered orders — log the drop

`order_delivered_customer_date IS NULL` marks orders that never completed the delivery path (still in transit or cancelled). We drop them before any analysis and report the count for the decisions log.

In [6]:
from olist.pipeline.demand import filter_delivered

orders = tables["orders"]
orders_delivered = filter_delivered(orders)
dropped = orders.count() - orders_delivered.count()
print(f"orders dropped (not-yet-delivered / cancelled): {dropped:,}")
print(f"orders retained: {orders_delivered.count():,}")


orders dropped (not-yet-delivered / cancelled): 2,965


orders retained: 96,476


## 6. Hot DataFrame — 4-way order-line join, cached via `@step`

`orders_delivered ⋈ order_items ⋈ broadcast(sellers) ⋈ broadcast(products)` with derived `delivery_delay_days`, `purchase_date`, `year_week`. Small lookups are broadcast per CLAUDE.md §3. The parquet is written to `outputs/_cache/demand_order_lines.parquet` (gitignored); all downstream steps read from that parquet — deterministic lineage, no stale memory state.

In [7]:
from olist.pipeline.demand import build_order_lines

order_lines = build_order_lines(spark)
print("order_lines rows:", order_lines.count())
order_lines.printSchema()


26/04/23 13:34:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[cache] run demand.order_lines (3.4s, wrote 110196 rows → outputs/_cache/demand_order_lines.parquet)
order_lines rows: 110196
root
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- product_category_name: strin

## 7. SparkSQL — three queries on a temp view

Registers `order_lines` as a temp view and runs the three queries required by the rubric. Each query's SQL text is printed alongside the result so the grader can read both. (1) Top 10 sellers by total revenue, (2) weekly order-volume preview, (3) late-delivery rate by state.

In [8]:
from olist.pipeline.demand import sparksql_queries

queries = sparksql_queries(order_lines)
for name, (sql, result) in queries.items():
    print(f"\n--- {name} ---")
    print(sql.strip())
    result.show(truncate=False)



--- top_sellers_by_revenue ---
SELECT seller_id,
           seller_state,
           ROUND(SUM(price), 2) AS total_revenue,
           COUNT(*)             AS line_count
    FROM order_lines
    GROUP BY seller_id, seller_state
    ORDER BY total_revenue DESC
    LIMIT 10


+--------------------------------+------------+-------------+----------+
|seller_id                       |seller_state|total_revenue|line_count|
+--------------------------------+------------+-------------+----------+
|4869f7a5dfa277a7dca6462dcf3b52b2|SP          |226987.93    |1148      |
|53243585a1d6dc2643021fd1853d8905|BA          |217940.44    |400       |
|4a3ca9315b744ce9f8e9374361493884|SP          |196882.12    |1949      |
|fa1c13f2614d7b5c4749cbc52fecda94|SP          |190917.14    |579       |
|7c67e1448b00f6e969d365cea6b010ab|SP          |186570.05    |1355      |
|7e93a43ef30c4f03f38b393420bc753a|SP          |165981.49    |322       |
|da8622b14eb17ae2831f4ac5b9dab84a|SP          |159816.87    |1548      |
|7a67c85e85bb2ce8582c35f2203ad736|SP          |139658.69    |1155      |
|1025f0e2d44d7041d6cf58b6550e0bfa|SP          |138208.56    |1420      |
|955fee9216a65b617aa5c0531780ce60|SP          |131836.71    |1472      |
+--------------------------------+------------+----

+--------------------------------+---------+------------------+
|seller_id                       |year_week|weekly_order_count|
+--------------------------------+---------+------------------+
|0015a82c2db000af6aaaf3ae2ecb0532|2017-39  |1                 |
|0015a82c2db000af6aaaf3ae2ecb0532|2017-41  |1                 |
|0015a82c2db000af6aaaf3ae2ecb0532|2017-42  |1                 |
|001cca7ae9ae17fb1caed9dfb1094831|2017-05  |1                 |
|001cca7ae9ae17fb1caed9dfb1094831|2017-07  |5                 |
|001cca7ae9ae17fb1caed9dfb1094831|2017-08  |1                 |
|001cca7ae9ae17fb1caed9dfb1094831|2017-09  |7                 |
|001cca7ae9ae17fb1caed9dfb1094831|2017-11  |4                 |
|001cca7ae9ae17fb1caed9dfb1094831|2017-12  |6                 |
|001cca7ae9ae17fb1caed9dfb1094831|2017-13  |5                 |
+--------------------------------+---------+------------------+


--- late_rate_by_state ---
SELECT seller_state,
           ROUND(AVG(CASE WHEN delivery_delay_days > 0

+------------+---------+--------------+-------+
|seller_state|late_rate|avg_delay_days|n_lines|
+------------+---------+--------------+-------+
|AM          |0.3333   |9.0           |3      |
|MA          |0.1940   |-11.26        |402    |
|RN          |0.0714   |-13.48        |56     |
|SP          |0.0711   |-11.3         |78600  |
|RJ          |0.0691   |-12.51        |4689   |
|CE          |0.0667   |-13.38        |90     |
|MS          |0.0600   |-17.38        |50     |
|DF          |0.0600   |-13.18        |883    |
|ES          |0.0577   |-13.36        |364    |
|PR          |0.0530   |-14.23        |8487   |
|SC          |0.0485   |-14.2         |4000   |
|MG          |0.0479   |-13.47        |8602   |
|BA          |0.0449   |-12.81        |624    |
|MT          |0.0417   |-15.63        |144    |
|PE          |0.0337   |-16.24        |445    |
+------------+---------+--------------+-------+



## 8. EDA stats — `approxQuantile` + `approx_count_distinct`

Both are big-data-safe: quantiles are computed on a sketch (here 1% relative error), and `approx_count_distinct` is a HyperLogLog sketch — neither requires a full shuffle. `.first()` on the single-row aggregate is the idiomatic Spark pattern for pulling a scalar to the driver.

In [9]:
from olist.pipeline.demand import eda_stats

stats = eda_stats(order_lines)
print("price quantiles (p25/p50/p75/p95):", stats["price_quantiles"])
print("delay quantiles (p25/p50/p75/p95):", stats["delay_quantiles"])
stats["approx_counts"].show()


price quantiles (p25/p50/p75/p95): [39.9, 72.9, 129.99, 310.0]
delay quantiles (p25/p50/p75/p95): [-17.0, -13.0, -8.0, 1.0]


+--------------+---------------+
|approx_sellers|approx_products|
+--------------+---------------+
|          2965|          32138|
+--------------+---------------+



## 9. Weekly order volume → parquet (consumed by NB2)

Aggregate `order_lines` to `(seller_id, year_week, weekly_order_count)` and repartition by `seller_id` so NB2's lead-indicator join co-locates partitions. Written to `outputs/nb1_weekly_order_volume.parquet` — a committed artefact.

In [10]:
from olist.pipeline.demand import build_weekly_order_volume
from pyspark.sql import functions as F

weekly_order_volume = build_weekly_order_volume(spark)
print("weekly rows:", weekly_order_volume.count())
weekly_order_volume.orderBy("seller_id", "year_week").limit(5).show()


[cache] run demand.weekly_order_volume (0.7s, wrote 35385 rows → outputs/nb1_weekly_order_volume.parquet)
weekly rows: 35385
+--------------------+---------+------------------+
|           seller_id|year_week|weekly_order_count|
+--------------------+---------+------------------+
|0015a82c2db000af6...|  2017-39|                 1|
|0015a82c2db000af6...|  2017-41|                 1|
|0015a82c2db000af6...|  2017-42|                 1|
|001cca7ae9ae17fb1...|  2017-05|                 1|
|001cca7ae9ae17fb1...|  2017-07|                 5|
+--------------------+---------+------------------+



## 10. Feature engineering Pipeline — lag / rolling / calendar features

The Pipeline has two stages: `Imputer` (median-imputation of `lag_1`, `lag_4`, `rolling_4w_mean`) then `VectorAssembler` (the six model features). `Window.partitionBy(seller_id).orderBy(year_week)` provides the lag + rolling primitives. The stages are printed inline so the grader can see what the Pipeline encapsulates.

In [11]:
from olist.pipeline.demand import build_feature_pipeline, add_weekly_features, FEATURE_COLS

feature_pipeline = build_feature_pipeline()
print("Pipeline stages:")
for stage in feature_pipeline.getStages():
    print(" ", stage)
print("\nFeature columns:", FEATURE_COLS)


Pipeline stages:
  Imputer_283ca8284db7
  VectorAssembler_6641bce94e9c

Feature columns: ['week_num', 'lag_1', 'lag_4', 'rolling_4w_mean', 'month', 'is_q4']


In [12]:
weekly_features = add_weekly_features(weekly_order_volume)
print("weekly_features rows:", weekly_features.count())
weekly_features.select(
    "seller_id", "year_week", "weekly_order_count", *FEATURE_COLS
).limit(5).show()


weekly_features rows: 35385


+--------------------+---------+------------------+--------+-----+-----+---------------+-----+-----+
|           seller_id|year_week|weekly_order_count|week_num|lag_1|lag_4|rolling_4w_mean|month|is_q4|
+--------------------+---------+------------------+--------+-----+-----+---------------+-----+-----+
|0015a82c2db000af6...|  2017-39|                 1|       1|    2|    2|            2.0|   39|    1|
|0015a82c2db000af6...|  2017-41|                 1|       2|    1|    2|            1.0|   41|    1|
|0015a82c2db000af6...|  2017-42|                 1|       3|    1|    2|            1.0|   42|    1|
|001cca7ae9ae17fb1...|  2017-05|                 1|       1|    2|    2|            2.0|    5|    0|
|001cca7ae9ae17fb1...|  2017-07|                 5|       2|    1|    2|            1.0|    7|    0|
+--------------------+---------+------------------+--------+-----+-----+---------------+-----+-----+



## 11. MLlib — `GBTRegressor` + `RandomForestRegressor` under `CrossValidator(folds=3)`

Both estimators are wrapped in a `CrossValidator(numFolds=3, parallelism=2, seed=42)` and evaluated with `RegressionEvaluator(metricName='rmse')`. The `@step`-wrapped `fit_and_score` step fits both, picks the lower-test-RMSE winner, and writes three parquets: `demand_predictions` (full-model predictions), `demand_metrics` (one-row summary), and the committed `nb1_seller_demand_scores` artefact. The CV configuration is printed inline via `explainParams()`.

In [13]:
from pyspark.ml.evaluation import RegressionEvaluator
from olist.pipeline.demand import build_cv_estimators

evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
cv = build_cv_estimators(evaluator)
print("GBT CrossValidator:")
print(" ", cv["gbt_cv"].explainParams().splitlines()[:4])
print("numFolds:", cv["gbt_cv"].getNumFolds(), "| param grid size:", len(cv["gbt_cv"].getEstimatorParamMaps()))
print("RF CrossValidator:")
print("numFolds:", cv["rf_cv"].getNumFolds(), "| param grid size:", len(cv["rf_cv"].getEstimatorParamMaps()))


GBT CrossValidator:
  ['collectSubModels: Param for whether to collect a list of sub-models trained during tuning. If set to false, then only the single best sub-model will be available after fitting. If set to true, then all sub-models will be available. Warning: For large models, collecting all sub-models can cause OOMs on the Spark driver. (default: False)', 'estimator: estimator to be cross-validated (current: GBTRegressor_b8dae7d9381c)', "estimatorParamMaps: estimator param maps (current: [{Param(parent='GBTRegressor_b8dae7d9381c', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 means 1 leaf node; depth 1 means 1 internal node + 2 leaf nodes. Must be in range [0, 30].'): 3, Param(parent='GBTRegressor_b8dae7d9381c', name='stepSize', doc='Step size (a.k.a. learning rate) in interval (0, 1] for shrinking the contribution of each estimator.'): 0.1}, {Param(parent='GBTRegressor_b8dae7d9381c', name='maxDepth', doc='Maximum depth of the tree. (>= 0) E.g., depth 0 me

In [14]:
from olist.pipeline.demand import fit_and_score

scoring = fit_and_score(spark)
print("--- demand_metrics ---")
scoring["demand_metrics"].show()

print("--- top-5 sellers by forecast_uplift_pct ---")
scoring["nb1_seller_demand_scores"].orderBy(
    F.col("forecast_uplift_pct").desc()
).limit(5).show(truncate=False)

print("--- demand_predictions preview ---")
scoring["demand_predictions"].orderBy("seller_id", "year_week").limit(5).show()


Fitting GBT CV ...


26/04/23 13:35:33 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Fitting RF CV ...


26/04/23 13:37:21 WARN DAGScheduler: Broadcasting large task binary with size 1278.6 KiB


26/04/23 13:37:23 WARN DAGScheduler: Broadcasting large task binary with size 2.0 MiB


...

26/04/23 13:38:22 WARN DAGScheduler: Broadcasting large task binary with size 2.1 MiB


26/04/23 13:38:26 WARN DAGScheduler: Broadcasting large task binary with size 3.3 MiB


GBT  test RMSE: 5.094
RF   test RMSE: 5.060
Selected: RandomForest


[cache] run demand.fit_and_score (231.7s, wrote 38356 rows → outputs/_cache/demand_predictions.parquet)
--- demand_metrics ---
+-----------------+-----------------+------------+
|         gbt_rmse|          rf_rmse|   best_name|
+-----------------+-----------------+------------+
|5.094275909684417|5.059765019604484|RandomForest|
+-----------------+-----------------+------------+

--- top-5 sellers by forecast_uplift_pct ---
+--------------------------------+------------+-------------------+-------------------+---------------+
|seller_id                       |seller_state|forecast_uplift_pct|avg_delay_days     |delay_risk_flag|
+--------------------------------+------------+-------------------+-------------------+---------------+
|6f892e20a171e98efe17fdb971ff319b|SP          |339.4886446907762  |-10.14             |0              |
|6973a06f484aacf400ece213dbf3d946|SP          |332.6384575054073  |-7.453703703703703 |0              |
|b8bc237ba3788b23da09c0f1f3a3288c|SC          |179.7

+--------------------+---------+--------+-----+------------------+
|           seller_id|year_week|week_num|label|        prediction|
+--------------------+---------+--------+-----+------------------+
|0015a82c2db000af6...|  2017-39|       1|  1.0|1.6254931726847766|
|0015a82c2db000af6...|  2017-41|       2|  1.0| 1.630966524807215|
|0015a82c2db000af6...|  2017-42|       3|  1.0|1.6436334282510974|
|001cca7ae9ae17fb1...|  2017-05|       1|  1.0|1.6978557296217087|
|001cca7ae9ae17fb1...|  2017-07|       2|  5.0|1.7191818044509457|
+--------------------+---------+--------+-----+------------------+



## 12. Artefacts produced + clean up

Committed parquets (read by NB2 / NB3 / `00_main.ipynb`):
- `outputs/geo_centroids.parquet`
- `outputs/nb1_weekly_order_volume.parquet`
- `outputs/nb1_seller_demand_scores.parquet`

Cache-only parquets (gitignored under `outputs/_cache/`):
- `demand_order_lines.parquet`, `demand_predictions.parquet`, `demand_metrics.parquet`

In [15]:
committed_outputs = sorted((ROOT_OUT := Path.cwd().parent / "outputs" if Path.cwd().name == "notebooks" else Path.cwd() / "outputs").glob("*.parquet"))
for path in committed_outputs:
    print(" ", path.name)
spark.stop()
print("\nSpark stopped.")


  geo_centroids.parquet
  nb1_seller_demand_scores.parquet
  nb1_weekly_order_volume.parquet
  nb2_seller_sentiment_scores.parquet
  nb3_seller_network_scores.parquet
  seller_risk_index.parquet



Spark stopped.
